# OCR + Laya in Google Colab

Runs the `ocr-platform` experimental crop flow — **PaddleOCR recognition → deterministic
validation → Laya occupation classification → human review** — entirely in Colab.
No local setup, no Windows, no PowerShell.

| Step | What happens |
|---|---|
| 1. Setup | Clone `Sisoodiya/ocr-platform@codex/paddle-laya-document-trial` |
| 2. Install | `pip install -e ".[dev,flow]"` (PaddleOCR, PaddlePaddle, Laya) |
| 3. Models | Download the Devanagari recognition model + the Laya multilingual checkpoint |
| 4. Smoke test | `flow.cli demo` — simulated, proves the wiring |
| 5. Real run | A synthetic Devanagari survey sheet through **real** PaddleOCR + Laya |
| 6. Review | Record a human correction and watch the provenance chain update |
| 7. Your PDFs | Look at the real WARD-42 scans |

**Runtime:** Python 3.12 · PaddleOCR on CPU · Laya on GPU when a CUDA device is present
(falls back to CPU). Expect ~5–10 min for the install and ~1 min for the models.

**Read first:** this is the *experimental annotated-crop* flow. It does not locate table
rows, register templates, or extract whole households. Every model decision starts in
human review with a null final value — auto-accept is **off**. See `docs/flow.md`.


In [1]:
import platform
import shutil
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/ocr-platform")
BRANCH = "codex/paddle-laya-document-trial"


def run_cli(*args: str) -> subprocess.CompletedProcess:
    """Run the ocr-platform flow CLI from the repo root."""
    return subprocess.run(
        [sys.executable, "-m", "ocr_platform.flow.cli", *args],
        cwd=REPO,
    )


print("python  :", sys.version.split()[0])
print("platform:", platform.platform())
print("disk free on /content: %.1f GB" % (shutil.disk_usage("/content").free / 1e9))

try:
    import torch

    print("torch   :", torch.__version__, "| cuda:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu     :", torch.cuda.get_device_name(0))
except ImportError:
    print("torch   : not installed yet")


python  : 3.13.16
platform: Linux-6.6.122+-x86_64-with-glibc2.39
disk free on /content: 93.5 GB
torch   : 2.11.0+cpu | cuda: False


## 1 · Clone the repo

The notebook needs the package source. We clone the branch that carries the
PaddleOCR → Laya document trial.


In [2]:
if (REPO / "pyproject.toml").is_file():
    print("repo already present:", REPO)
else:
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            BRANCH,
            "https://github.com/Sisoodiya/ocr-platform.git",
            str(REPO),
        ],
        check=True,
    )

branch = subprocess.run(
    ["git", "rev-parse", "--abbrev-ref", "HEAD"],
    cwd=REPO,
    capture_output=True,
    text=True,
).stdout.strip()
print("cloned branch:", branch)


cloned branch: codex/paddle-laya-document-trial


## 2 · Install

`configs/flow_trial_constraints.txt` pins the exact versions used by the
2026-10-05 document trial. We drop the `torch` line so Colab's preinstalled
CUDA build of torch stays untouched for Laya.


In [3]:
lines = [
    line.strip()
    for line in (REPO / "configs" / "flow_trial_constraints.txt").read_text().splitlines()
    if line.strip()
    and not line.strip().startswith("#")
    and not line.strip().startswith("torch==")
]
Path("/tmp/flow_colab_constraints.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))


paddleocr==3.7.0
paddlepaddle==3.3.1
paddlex==3.7.2
laya==0.3.26
transformers==5.18.0
numpy==2.3.5
opencv-contrib-python==4.10.0.84


In [4]:
# paddlex requires opencv-contrib-python; the two opencv builds cannot coexist.
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "-q",
     "opencv-python", "opencv-python-headless"],
    check=False,
)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "-e", ".[dev,flow]", "-c", "/tmp/flow_colab_constraints.txt"],
    cwd=REPO,
    check=True,
)
print("install finished")


install finished


In [5]:
import importlib.metadata as md

for name in (
    "paddleocr",
    "paddlepaddle",
    "paddlex",
    "laya",
    "torch",
    "transformers",
    "pypdfium2",
    "numpy",
    "opencv-contrib-python",
):
    try:
        print(f"{name:24s} {md.version(name)}")
    except md.PackageNotFoundError:
        print(f"{name:24s} MISSING")


paddleocr                3.7.0
paddlepaddle             3.3.1
paddlex                  3.7.2
laya                     0.3.26
torch                    2.11.0+cpu
transformers             5.18.0
pypdfium2                5.14.0
numpy                    2.3.5
opencv-contrib-python    4.10.0.84


## 3 · Download the models

* **PaddleOCR** — `devanagari_PP-OCRv5_mobile_rec` inference weights. The flow's
  `TextRecognition(model_dir=...)` needs `inference.json`, `inference.pdiparams`
  and `inference.yml` in one directory.
* **Laya** — `convaiinnovations/laya-multilingual` at the publisher revision used by the
  2026-10-05 trial. Needs `rl_agent_config.json`, `model.safetensors`, `tokenizer/`
  and `encoder/config.json`.

The CLI sets `HF_HUB_OFFLINE=1` before loading, so anything not already on disk will
fail rather than download silently.


In [6]:
from huggingface_hub import snapshot_download

PADDLE_DIR = REPO / "models" / "devanagari_PP-OCRv5_mobile_rec_infer"
LAYA_DIR = REPO / "models" / "laya-multilingual"

if (PADDLE_DIR / "inference.pdiparams").is_file():
    print("paddle model already present")
else:
    snapshot_download(
        "PaddlePaddle/devanagari_PP-OCRv5_mobile_rec",
        local_dir=str(PADDLE_DIR),
    )
    print("paddle model ->", PADDLE_DIR)

if (LAYA_DIR / "model.safetensors").is_file():
    print("laya model already present")
else:
    snapshot_download(
        "convaiinnovations/laya-multilingual",
        revision="e4e9ddf21a7b1903b7acffd8814ad4307bf63a67",
        local_dir=str(LAYA_DIR),
        allow_patterns=["rl_agent_config.json", "model.safetensors", "tokenizer/*", "encoder/*"],
    )
    print("laya model ->", LAYA_DIR)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

paddle model -> /content/ocr-platform/models/devanagari_PP-OCRv5_mobile_rec_infer


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

laya model -> /content/ocr-platform/models/laya-multilingual


In [7]:
required = {
    PADDLE_DIR: ["inference.json", "inference.pdiparams", "inference.yml"],
    LAYA_DIR: ["rl_agent_config.json", "model.safetensors", "encoder/config.json"],
}
for folder, names in required.items():
    for name in names:
        path = folder / name
        assert path.is_file(), f"MISSING: {path}"
        print(f"OK  {path.relative_to(REPO)}  ({path.stat().st_size / 1e6:.1f} MB)")
assert (LAYA_DIR / "tokenizer").is_dir(), "MISSING: tokenizer/"
print("OK  tokenizer/")


OK  models/devanagari_PP-OCRv5_mobile_rec_infer/inference.json  (0.2 MB)
OK  models/devanagari_PP-OCRv5_mobile_rec_infer/inference.pdiparams  (7.8 MB)
OK  models/devanagari_PP-OCRv5_mobile_rec_infer/inference.yml  (0.0 MB)
OK  models/laya-multilingual/rl_agent_config.json  (0.0 MB)
OK  models/laya-multilingual/model.safetensors  (643.8 MB)
OK  models/laya-multilingual/encoder/config.json  (0.0 MB)
OK  tokenizer/


## 4 · Smoke test (no models)

`demo` uses a simulated recognizer and classifier. It proves the wiring — dataset
loading, rendering, validation, classification routing, review queue — without
touching a model. It is **not** accuracy evidence.


In [8]:
result = run_cli("demo", "--out", "runs/flow-demo")
print("exit code:", result.returncode)


exit code: 0


In [9]:
import json

summary_path = next((REPO / "runs" / "flow-demo").rglob("summary.json"))
print(json.dumps(json.loads(summary_path.read_text()), indent=2))


{
  "accepted": 0,
  "classified": 1,
  "classifier_failures": 0,
  "fields": 4,
  "ocr_failures": 0,
  "output": "/content/ocr-platform/runs/flow-demo/flow/fb75e9119ae79fce83b80d30d1cfc2ee460240da129f8b7e31de5939dc246737",
  "review": 4,
  "run_id": "fb75e9119ae79fce83b80d30d1cfc2ee460240da129f8b7e31de5939dc246737",
  "scope": "experimental annotated-crop flow; production auto-accept disabled"
}


## 5 · A real sheet through real models

Now we build a small survey sheet with **real printed Devanagari and Latin text** and
run it through the actual PaddleOCR recognizer and the actual Laya classifier.

The page is rendered at 200 dpi and saved as a PDF at the same resolution, so the
flow's own 200 dpi render is pixel-for-pixel identical to what we drew. Bounding
boxes are captured from the font's own text metrics.


In [11]:
import urllib.request

from PIL import features

FONT_PATH = REPO / "models" / "NotoSansDevanagari-Regular.ttf"
if not FONT_PATH.is_file():
    FONT_PATH.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(
        "https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/"
        "NotoSansDevanasari/NotoSansDevanasari-Regular.ttf",
        FONT_PATH,
    )
print("font:", FONT_PATH, f"({FONT_PATH.stat().st_size / 1e6:.1f} MB)")
print("raqm (complex-text shaping):", features.check("raqm"))


HTTPError: HTTP Error 404: Not Found

In [ ]:
import hashlib
import json

from PIL import Image, ImageDraw, ImageFont

WIDTH, HEIGHT = 1600, 1200
DPI = 200  # the flow renders at 200 dpi by default

font = ImageFont.truetype(str(FONT_PATH), 56)
label_font = ImageFont.truetype(str(FONT_PATH), 40)

# (field_id, printed label, answer value)
ROWS = [
    ("document.ward", "Ward", "42"),
    ("member.row1.name", "Name", "आरती शर्मा"),
    ("member.row1.age", "Age", "34"),
    ("member.row1.gender", "Gender", "M"),
    ("member.row1.religion", "Religion", "H"),
    ("member.row1.occupation", "Occupation", "दर्ज़ी"),
    ("member.row2.occupation", "Occupation", "किसान"),
    ("member.row3.occupation", "Occupation", "teacher"),
]

page = Image.new("RGB", (WIDTH, HEIGHT), "white")
draw = ImageDraw.Draw(page)

fields = []
y = 60
for field_id, label, value in ROWS:
    draw.text((80, y), label, font=label_font, fill="#333333")
    x0 = 520
    draw.text((x0, y), value, font=font, fill="black")
    left, top, right, bottom = draw.textbbox((x0, y), value, font=font)
    pad = 12
    bbox = [
        (left - pad) / WIDTH,
        (top - pad) / HEIGHT,
        (right + pad) / WIDTH,
        (bottom + pad) / HEIGHT,
    ]
    fields.append(
        {
            "id": field_id,
            "page": 0,
            "bbox": bbox,
            "kind": "print",
            "state": "present",
            "expected": value,
            "labeler": "colab-synthetic-generator",
        }
    )
    y += 130

source_dir = REPO / "runs" / "colab-demo" / "source"
source_dir.mkdir(parents=True, exist_ok=True)
pdf_path = source_dir / "synthetic-survey.pdf"
page.save(pdf_path, format="PDF", resolution=DPI)
page.close()

pdf_bytes = pdf_path.read_bytes()
dataset = {
    "schema_version": 1,
    "source_root": str(source_dir.resolve()),
    "documents": [
        {
            "id": "synthetic-survey",
            "path": pdf_path.name,
            "sha256": hashlib.sha256(pdf_bytes).hexdigest(),
            "page_count": 1,
            "classification": "synthetic",
            "split": "development",
            "fields": fields,
        }
    ],
}
dataset_path = REPO / "runs" / "colab-demo" / "dataset.json"
dataset_path.write_text(json.dumps(dataset, ensure_ascii=False, indent=2), encoding="utf-8")

print("pdf sha256:", dataset["documents"][0]["sha256"])
print("dataset  :", dataset_path)
print()
for field in fields:
    box = [round(v, 4) for v in field["bbox"]]
    print(f"  {field['id']:28s} {field['expected']!r:16s} {box}")


## 6 · Run the real flow

`flow.cli run` renders the PDF, crops each answer region, runs PaddleOCR
recognition, applies the deterministic validation rules, and sends occupation text
to Laya. Everything lands in the review queue.

Laya runs on PyTorch — on GPU when Colab gives us one, otherwise CPU.


In [ ]:
import torch

laya_device = "cuda" if torch.cuda.is_available() else "cpu"
print("paddle device: cpu")
print("laya  device:", laya_device)


In [ ]:
result = run_cli(
    "run",
    "--dataset", "runs/colab-demo/dataset.json",
    "--paddle-model-dir", "models/devanagari_PP-OCRv5_mobile_rec_infer",
    "--paddle-model-name", "devanagari_PP-OCRv5_mobile_rec",
    "--paddle-device", "cpu",
    "--laya-model-dir", "models/laya-multilingual",
    "--laya-device", laya_device,
    "--out", "runs/colab-flow",
)
print("exit code:", result.returncode, "(1 means some fields failed and stayed in review)")


In [ ]:
import json
from pathlib import Path

run_dir = next((REPO / "runs" / "colab-flow").rglob("summary.json")).parent
summary = json.loads((run_dir / "summary.json").read_text())
print(json.dumps(summary, indent=2))

decisions = [
    json.loads(line)
    for line in (run_dir / "decisions.jsonl").read_text().splitlines()
]

print()
header = f"{'field_id':30s} {'ocr_text':14s} {'conf':>6s}  {'category':16s} reasons"
print(header)
print("-" * len(header))
for decision in decisions:
    record = decision["record"]
    classification = record["classification"] or {}
    confidence = record["ocr_confidence"]
    shown = "n/a" if confidence is None else f"{confidence:.3f}"
    print(
        f"{record['field_id']:30s} {record['raw_text'][:13]:14s} "
        f"{shown:>6s}  {str(classification.get('category'))[:15]:16s} "
        f"{','.join(record['reason_codes'])}"
    )


In [ ]:
# Show the crop the model actually saw for the first occupation field.
from PIL import Image

occupation = next(
    d for d in decisions
    if d["record"]["field_id"].endswith("occupation")
)
crop = run_dir / "crops" / f"{occupation['record']['evidence']['crop_sha256']}.png"
print("crop:", crop.name)
Image.open(crop)


## 7 · Record a human correction

Every decision starts in review with a null final value. A human correction supplies
the accepted value (and, for occupations, the accepted category), is hash-chained to
the original decision, and rebuilds the effective results.

Because we generated the sheet, we know the ground truth — so this is a real
correction, not a guess.


In [ ]:
import json

dataset = json.loads(
    (REPO / "runs" / "colab-demo" / "dataset.json").read_text(encoding="utf-8")
)
expected = {f["id"]: f["expected"] for f in dataset["documents"][0]["fields"]}

candidates = json.loads(
    (run_dir / "review_candidates.json").read_text(encoding="utf-8")
)["candidates"]
print(f"{len(candidates)} decision(s) awaiting human review\n")
for candidate in candidates:
    record = candidate["record"]
    print(f"  {candidate['decision_id'][:16]}  {record['field_id']:28s} ocr={record['raw_text']!r}")


In [ ]:
import json

CATEGORY_BY_TEXT = {
    "दर्ज़ी": "SELF_EMPLOYED",  # tailor
    "किसान": "AGRICULTURE",  # farmer
    "teacher": "OTHER",
}

target = next(c for c in candidates if c["record"]["field_id"].endswith("occupation"))
field_id = target["record"]["field_id"]
value = expected[field_id]
category = CATEGORY_BY_TEXT[value]

print(f"correcting {field_id}")
print(f"  ocr read : {target['record']['raw_text']!r}")
print(f"  human set: {value!r}  ->  {category}\n")

result = run_cli(
    "review",
    "--run", str(run_dir),
    "--decision-id", target["decision_id"],
    "--value", value,
    "--category", category,
    "--reviewer", "colab-reviewer",
)
print("exit code:", result.returncode)


In [ ]:
import json

summary = json.loads((run_dir / "summary.json").read_text())
print(
    json.dumps(
        {
            key: summary[key]
            for key in ("fields", "review", "accepted", "classified",
                        "ocr_failures", "classifier_failures")
        },
        indent=2,
    )
)
revisions = (run_dir / "human_revisions.jsonl").read_text().splitlines()
print(f"\n{len(revisions)} human revision(s) recorded")


## 8 · Your own WARD-42 PDFs

You have 1,446 two-page scanned WARD-42 PDFs. Two ways to get them into Colab.

**Option A — upload a zip** (simplest). On your machine:

```bash
cd /home/roshan/Documents
zip -r WARD-42.zip WARD-42
```

then run the cell below and pick `WARD-42.zip`.

**Option B — mount Google Drive** if the PDFs are already there.

> The flow needs a `dataset.json` with answer-region boxes. Producing verified labels
> is a human task — use the benchmark labeler (`src/ocr_platform/benchmark/labeler.html`)
> locally, or `scripts/prepare_document_trial.py` with `configs/document_trial_regions.json`.
> The cell after this one shows raw PaddleOCR output on a real page without needing
> any labels.


In [ ]:
import subprocess
import zipfile
from pathlib import Path

TARGET = Path("/content/WARD-42")

if TARGET.is_dir() and any(TARGET.rglob("*.pdf")):
    print("WARD-42 already present")
else:
    from google.colab import files

    uploaded = files.upload()
    archive = Path("/content") / next(iter(uploaded))
    print("uploaded:", archive.name, f"({archive.stat().st_size / 1e6:.1f} MB)")
    with zipfile.ZipFile(archive) as zf:
        zf.extractall("/content")
    archive.unlink()

pdfs = sorted(TARGET.rglob("*.pdf"))
print(f"{len(pdfs)} PDFs under {TARGET}")
for path in pdfs[:3]:
    print("  ", path.relative_to(TARGET))


In [ ]:
# Raw PaddleOCR on a real WARD-42 page -- no labels, no dataset, no Laya.
# Renders page 1 at 200 dpi (exactly what the flow does) and runs detection +
# recognition, so you can see what the engine actually reads on your scans.
import numpy as np
import pypdfium2 as pdfium
from PIL import Image, ImageDraw

from paddleocr import PaddleOCR

PDF = pdfs[0]
PAGE_INDEX = 0
DPI = 200

with pdfium.PdfDocument(str(PDF)) as document:
    page = document[PAGE_INDEX]
    bitmap = page.render(scale=DPI / 72)
    image = bitmap.to_pil().copy()
    bitmap.close()
    page.close()

print("page:", PDF.name, "page", PAGE_INDEX + 1, "rendered:", image.size)

ocr = PaddleOCR(
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
    text_recognition_model_name="devanagari_PP-OCRv5_mobile_rec",
    device="cpu",
)

result = next(ocr.predict(input=np.asarray(image)))
data = result.json

print(f"\n{len(data['rec_texts'])} text line(s) detected\n")
for text, score in zip(data["rec_texts"], data["rec_scores"]):
    print(f"  {float(score):.3f}  {text!r}")

annotated = image.copy()
draw = ImageDraw.Draw(annotated)
width, height = image.size
for poly, text in zip(data["dt_polys"], data["rec_texts"]):
    points = [(float(p[0]), float(p[1])) for p in poly]
    draw.line(points + [points[0]], fill=(220, 30, 30), width=3)
annotated


## Caveats

* **Draft rules.** `configs/flow_rulesv1.json` is explicitly a draft. The occupation
  categories and the 0.8 confidence threshold are illustrative, not a survey codebook.
* **Auto-accept is off.** Nothing is ever accepted without a human revision. High
  confidence does not change that.
* **No accuracy claim.** The 2026-10-05 trial found numeric misreads (48 read as 98,
  24 read as 2 at ~0.93 confidence) and Laya assigned `OTHER` to every printed
  employment control. Treat every value as unverified.
* **Handwriting is unproven.** The trial's five handwritten occupations all failed the
  OCR gate and never reached Laya. Printed Devanagari support does not establish
  handwritten accuracy.
* **Scope.** This is the annotated-crop experiment. Whole-form extraction, row
  location and template registration are not implemented.

See `docs/flow.md`, `docs/test-results/document-trial-2026-10-05.md` and
`docs/test-results/fine-tuning-pilot-2026-10-05.md` for the full picture.
